# Week 7 Common Theory Assessment · 30 Marks

**Student name:** DAKSHINI ANAND NEEL  
**Student ID:** 2026CS001  
**Department:** Computer Engineering  
**Program:** MindForgeAI Internship 2026  

This notebook was generated from the authenticated Student Portal.

## Instructions

Answer all six questions individually in the answer cells provided. Use concise reasoning, equations, tables, diagrams or small code demonstrations where useful. Do not delete the question cells. Save and execute the notebook, then upload it to Colab, GitHub or Drive and add its shareable link to the Week 7 submission window.

## Q1 · Data splitting and leakage · 5 marks

Distinguish training, validation and test data. Explain two forms of data leakage and show how a pipeline prevents one of them.

#### 1. Distinguishing Data Splits
* **Training Data:** Used strictly to compute gradients and update model parameters (e.g., weights $w$ and biases $b$)[cite: 1].
* **Validation Data:** Used during training to evaluate model performance, perform hyperparameter tuning, and detect overfitting early[cite: 1].
* **Test Data:** A completely untouched holdout set evaluated strictly once at the end of the project to estimate true real-world generalization[cite: 1].

---

#### 2. Two Forms of Data Leakage
1. **Target Leakage:** Including features that inherently contain knowledge of the target variable which wouldn't be available at prediction time (e.g., including `days_in_hospital` when predicting if a patient *will* be admitted)[cite: 1].
2. **Preprocessing Leakage (Train-Test Contamination):** Calculating statistical parameters (like mean $\mu$, variance $\sigma^2$, or min/max boundaries) across the entire dataset prior to splitting[cite: 1].

---

#### 3. Preventing Preprocessing Leakage with `sklearn.pipeline.Pipeline`

A Scikit-Learn `Pipeline` encapsulates data transformations alongside the estimator, ensuring that transformations `.fit()` **only on the training fold** during cross-validation[cite: 1].

In [17]:
# ==============================================================================
# PRO-LEVEL VISUAL & CODE: Data Leakage Prevention via sklearn Pipeline
# ==============================================================================
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# The Pipeline ensures fit() is NEVER called on validation data during CV
ml_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Visualizing the Pipeline Information Flow (Safe from Leakage)
pipeline_visual = """
 ┌────────────────────────────────────────────────────────┐
 │ CROSS-VALIDATION SPLIT (e.g., Fold 1)                  │
 ├─────────────────────────┬──────────────────────────────┤
 │ TRAIN FOLD              │ VALIDATION FOLD              │
 │ 1. scaler.fit(train)    │                              │
 │ 2. scaler.transform()   │ 1. scaler.transform(val)     │
 │ 3. model.fit(scaled)    │ 2. model.predict(scaled)     │
 └─────────────────────────┴──────────────────────────────┘
    * Notice: scaler.fit() NEVER touches the Val Fold! *
"""
print(pipeline_visual)

# Assuming X, y are predefined features and targets
# cv_scores = cross_val_score(ml_pipeline, X, y, cv=5)


 ┌────────────────────────────────────────────────────────┐
 │ CROSS-VALIDATION SPLIT (e.g., Fold 1)                  │
 ├─────────────────────────┬──────────────────────────────┤
 │ TRAIN FOLD              │ VALIDATION FOLD              │
 │ 1. scaler.fit(train)    │                              │
 │ 2. scaler.transform()   │ 1. scaler.transform(val)     │
 │ 3. model.fit(scaled)    │ 2. model.predict(scaled)     │
 └─────────────────────────┴──────────────────────────────┘
    * Notice: scaler.fit() NEVER touches the Val Fold! *



In [18]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# 1. Synthesize dummy feature matrix and target
np.random.seed(42)
X = np.random.randn(200, 5)
y = np.random.randint(0, 2, size=200)

# 2. Encapsulate transformer and estimator inside a Pipeline
# The pipeline mathematically guarantees zero feature leakage across CV folds
ml_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(solver='lbfgs'))
])

# 3. Evaluate using K-Fold Cross-Validation safely
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(ml_pipeline, X, y, cv=cv, scoring='accuracy')

print(f"Leakage-Free CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

Leakage-Free CV Accuracy: 0.4950 ± 0.0430


## Q2 · Metrics under class imbalance · 5 marks

A classifier has 96% accuracy but the positive class is only 3% of records. Explain why accuracy may mislead. Define precision and recall, then choose a primary metric for a high-cost false-negative situation.

### **1. The Accuracy Paradox under Class Imbalance**
 When the positive class represents only 3% of the records, standard accuracy becomes highly deceptive. A "dummy" model that hardcodes a prediction of the majority class (negative) for every single record will achieve 97% accuracy. Even if a model claims 96% accuracy, it might be misclassifying every single positive instance. High accuracy here masks a totally useless model.  
### **2. Defining Precision and Recall**
**Precision:** The ratio of correctly predicted positive observations to the total predicted positives. It measures the cost of False Positives ($FP$).$$\text{Precision} = \frac{TP}{TP + FP}$$

**Recall (Sensitivity):** The ratio of correctly predicted positive observations to all actual positives. It measures the cost of False Negatives ($FN$).$$\text{Recall} = \frac{TP}{TP + FN}$$

### **3. Metric for High-Cost False Negatives**
In situations where a False Negative carries a massive cost (e.g., missing a cancer diagnosis or fraud detection), the primary metric must prioritize Recall. To evaluate the model holistically while heavily penalizing False Negatives, we use the $F_2$-Score, which weights Recall twice as heavily as Precision:
  $$F_2 = (1 + 2^2) \frac{\text{Precision} \cdot \text{Recall}}{(2^2 \cdot \text{Precision}) + \text{Recall}}$$

## Q3 · Cross-validation strategy · 5 marks

Explain K-fold cross-validation. State when ordinary K-fold is invalid or weak, and select suitable alternatives for (a) imbalanced classes, (b) repeated customers/devices and (c) time-ordered prediction.

### **1. Explaining K-Fold Cross-Validation**
In $K$-Fold CV, the dataset is randomly partitioned into $K$ mutually exclusive subsets (folds) of equal size. The model is trained $K$ times; iteratively, one fold is held out as the validation set while the remaining $K-1$ folds act as the training set.  

### **2. When Ordinary K-Fold Fails**
Standard K-Fold relies on the assumption of Independent and Identically Distributed (i.i.d.) data. It is fundamentally flawed when:  Target classes are heavily skewed (can result in zero minority samples in a fold).Records are logically grouped by an entity (leads to data leakage if the same entity is in train and validation).Data has a chronological order (random splitting allows future data to predict the past).

### **3. Suitable Alternatives**
**(a)Imbalanced classes:**
StratifiedKFold (Preserves the global class distribution ratio across all folds).

**(b) Repeated customers/devices:** GroupKFold (Ensures that all records belonging to a specific customer_id remain exclusively in either the train or validation fold, preventing leakage).

**(c) Time-ordered prediction:** TimeSeriesSplit (Uses an expanding window to respect temporal causality).

## Q4 · Bias, variance and generalisation · 5 marks

Explain underfitting and overfitting through bias and variance. Give four remedies, including at least one related to features and one related to validation/training.

### **1. Underfitting and Overfitting via Bias/Variance**
*   **Underfitting (High Bias):** The model is too simple to learn the data (e.g., trying to fit a linear line to a complex curve). It has a high error on both training and test data.
*   **Overfitting (High Variance):** The model is too complex and "memorizes" the noise in the training data (e.g., a high-degree polynomial that hits every point). It performs great on training data but fails on new, unseen data.

**The Error Decomposition Equation:**
$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

### **2. Four Remedies**
| Category | Remedy | Description |
| :--- | :--- | :--- |
| **Feature-related** | **Feature Selection** | Removing noisy or redundant features to reduce model complexity and variance. |
| **Validation-related**| **Cross-Validation** | Using K-Fold CV to ensure the model generalizes well across different subsets of data. |
| **Complexity-related**| **Regularization** | Adding a penalty (L1/L2) to the loss function to keep weight values small. |
| **Data-related** | **More Data** | Increasing the sample size helps the model identify the true signal over noise. |

## Q5 · Feature engineering · 5 marks

What is feature engineering? Give three valid examples and two leakage-producing examples. Explain why transformations must be fitted on training data only.

### **1. Defining Feature Engineering**
Feature engineering is using domain knowledge to transform raw data into features that help the algorithm perform better. It's essentially "highlighting" the most important patterns for the model.

### **2. Examples Table**
| Type | Example Description |
| :--- | :--- |
| **Valid** | 1. **Binning:** Grouping ages into 'Child', 'Adult', 'Senior'.<br>2. **Interaction:** Creating an `area` feature from `width * height`.<br>3. **Date Features:** Extracting 'Weekend vs Weekday' from a timestamp. |
| **Leakage** | 1. **Future Data:** Including 'Time of Death' to predict 'Cause of Death'.<br>2. **Global Stats:** Using the mean of the *whole* dataset to fill NaNs before the train/test split. |

### **3. Why Fit on Training Data Only?**
If we calculate scaling parameters (like Mean or Standard Deviation) using the test set, we are leaking information about the future data distribution into our training process.

```python
# Correct way to prevent leakage
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# FIT only on train, TRANSFORM both
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

## Q6 · Epochs, iterations and convergence · 5 marks

Compare epoch-based learning, solver iterations and one-call model fitting. Explain early stopping, patience and convergence. Why is it technically wrong to demand an epoch loop for every ML algorithm?

### **1. Comparison of Learning Approaches**
*   **Epoch-based:** Passes through the entire dataset multiple times (Common in Deep Learning).
*   **Solver Iterations:** Iterative optimization steps (e.g., Gradient Descent steps) to minimize loss.
*   **One-call Fitting:** Mathematical solutions solved in a single step (Closed-form).

### **2. Key Concepts**
*   **Convergence:** The state where the model has reached its best possible performance and the loss stops decreasing.
*   **Early Stopping:** Monitoring validation loss and stopping training early if it starts to rise.
*   **Patience:** The number of epochs we wait for an improvement before triggering Early Stopping.

### **3. Why an Epoch Loop isn't always correct**
Not all models "learn" by iterating over data multiple times. For example, Linear Regression can be solved analytically using the **Normal Equation**:
$$\hat{\beta} = (X^T X)^{-1} X^T y$$

```python
# Decision Trees don't use 'epochs' - they split data recursively
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train) # No epoch loop required!
```

## Submission checklist

- [X] Any external source is cited.
- [X] The notebook opens correctly and is shared with review access.
- [X] I added this notebook as a separately described link in the Week 7 submission.

## Academic integrity declaration

I, **DAKSHINI ANAND NEEL (2026CS001)**, declare that these answers represent my own understanding and that I can explain them during evaluation.

**Typed confirmation:** _Dakshini Anand Neel_